# 03_graphgrued_relabel_ordered5

This notebook **does not rebuild the raw ETL from scratch**. It reads the `ml.tbl_graphgrued_weekly_active_base` table created in `01_v2`, and then recreates the **5-level ordinal labels more aligned with the paper's spirit**: `A/B/C/D/E`, representing:

- **A**: excellent
- **B**: good
- **C**: satisfactory / average
- **D**: low-performing but still active, high risk
- **E**: incomplete / dropout

In addition to the string labels, the notebook also creates `risk_label_5_idx` in the order `A=0, B=1, C=2, D=3, E=4`.

In [1]:

BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base"
OUTPUT_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
LABEL_ORDER = ["A", "B", "C", "D", "E"]
print(BASE_TABLE, OUTPUT_TABLE)


StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 3, Finished, Available, Finished, False)

dbo.tbl_graphgrued_weekly_active_base dbo.tbl_graphgrued_weekly_active_base_ordered5


In [2]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table(BASE_TABLE)
print("rows:", df.count())
display(df.limit(5))


StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 4, Finished, Available, Finished, False)

rows: 100409


SynapseWidget(Synapse.DataFrame, 7da86ceb-70e0-4c72-b2bf-f958f73e8c3a)

In [3]:
display(df.summary())

StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0fb28cbc-5b1b-4bb8-b548-2dfd4c63bfbe)

In [4]:
display(
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
)

StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05e84f8b-5b2d-4371-aa61-375982320770)

In [5]:

# --- Helper columns to align with paper semantics ---
# course_norm_score: relative course performance proxy (0-1)
# completion_rate  : progress/completion proxy (0-1)
# activity_signal  : whether learner is still active in the observed 4-week window

activity_cols = [c for c in df.columns if c.endswith('_week1') or c.endswith('_week2') or c.endswith('_week3') or c.endswith('_week4')]
if not activity_cols:
    raise ValueError("No weekly columns found. Run 01_v2 first.")

df2 = df
df2 = df2.withColumn(
    'activity_signal_4w',
    sum(F.coalesce(F.col(c).cast('double'), F.lit(0.0)) for c in activity_cols)
)
df2 = df2.withColumn('is_active_4w', F.when(F.col('activity_signal_4w') > 0, F.lit(1)).otherwise(F.lit(0)))

# ordered 5-label strategy (paper-like surrogate)
# E: no activity or zero completion
# D: active but low score / low completion
# C/B/A: increasing score-completion quality
df2 = (
    df2
    .withColumn(
        'risk_label_5_name',
        F.when((F.col('is_active_4w') == 0) | (F.coalesce(F.col('completion_rate'), F.lit(0.0)) <= 0.01), F.lit('E'))
         .when((F.col('course_norm_score') >= 0.90) & (F.col('completion_rate') >= 0.90), F.lit('A'))
         .when((F.col('course_norm_score') >= 0.80) & (F.col('completion_rate') >= 0.80), F.lit('B'))
         .when((F.col('course_norm_score') >= 0.60) & (F.col('completion_rate') >= 0.60), F.lit('C'))
         .otherwise(F.lit('D'))
    )
    .withColumn(
        'risk_label_5_idx',
        F.when(F.col('risk_label_5_name') == 'A', F.lit(0))
         .when(F.col('risk_label_5_name') == 'B', F.lit(1))
         .when(F.col('risk_label_5_name') == 'C', F.lit(2))
         .when(F.col('risk_label_5_name') == 'D', F.lit(3))
         .otherwise(F.lit(4))
    )
    .withColumn('label_D_flag', F.when(F.col('risk_label_5_name') == 'D', F.lit(1)).otherwise(F.lit(0)))
    .withColumn('label_E_flag', F.when(F.col('risk_label_5_name') == 'E', F.lit(1)).otherwise(F.lit(0)))
)


StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 7, Finished, Available, Finished, False)

In [6]:

display(
    df2.groupBy('risk_label_5_name', 'risk_label_5_idx').count().orderBy('risk_label_5_idx')
)


StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3e8ebc3-5111-45af-9724-eaa4c42ab999)

In [7]:

(df2.write.mode('overwrite').format('delta').saveAsTable(OUTPUT_TABLE))
print('saved to', OUTPUT_TABLE)


StatementMeta(, 66e142b3-d0e6-4a1a-9dd8-b44cccef9b89, 9, Finished, Available, Finished, False)

saved to dbo.tbl_graphgrued_weekly_active_base_ordered5
